# Training and validation loss

`train_notes.py` records only the best validation loss, so the shipped runs left no
curve behind. This notebook retrains from scratch with the same hyperparameters and
keeps the per-epoch history instead.

**It writes no checkpoint.** `train_with_history.py` contains no `torch.save` call at
all, so nothing in `checkpoints/` can be touched — the model is discarded when the
run ends and only the loss history survives.

In [ ]:
import sys, pathlib, json

ROOT = pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))                                  # utils, model, train
sys.path.insert(0, str(ROOT / "experiments" / "loss-curves"))   # dir name has a hyphen,
                                                               # so import it by path
import numpy as np
import matplotlib.pyplot as plt
from train_with_history import train_with_history

%config InlineBackend.figure_formats = ['retina']

## 1. Train, or load an existing run

A 150-epoch bass run takes about 7.5 minutes on MPS. If the history file is already
there from a previous run, this loads it instead of retraining.

`--split augmented` reproduces `train_notes.py` exactly. `grouped` is the stricter
alternative described in section 3.

In [ ]:
HIST = ROOT / "experiments" / "loss-curves" / "history_bass_augmented.json"

if HIST.exists():
    hist = json.load(open(HIST))
    print(f"loaded existing run: {hist['epochs']} epochs, {hist['seconds']}s")
else:
    hist = train_with_history(role="bass", strategy="augmented", epochs=150)
    HIST.parent.mkdir(parents=True, exist_ok=True)
    json.dump(hist, open(HIST, "w"), indent=1)

print(f"best val {min(hist['val']):.4f} at epoch {int(np.argmin(hist['val'])) + 1}")
print(f"{hist['n_train']:,} train / {hist['n_val']:,} val rows")
print(f"{hist['shared_phrases']} of {hist['n_train_phrases']} phrases appear on both sides")

## 2. The plotting function

Two panels, because one doesn't work here. The first ~15 epochs drop the loss from
2.0 to 0.2, which flattens everything afterwards into a single line — so the left
panel shows the shape of the run and the right panel zooms on the converged region,
where the train/validation gap and the best-val epoch are actually legible.

Colours are the validated pair `#26699F` / `#C2542B` (CVD ΔE 17.7). The shaded band
between the curves is the generalisation gap.

In [ ]:
TRAIN, VAL = "#26699F", "#C2542B"      # validated pair: CVD dE 17.7, chroma + contrast pass
INK, MUTED, FAINT, RULE = "#16201F", "#63706E", "#8A9694", "#D5DCDA"


def _style(ax):
    ax.grid(color=RULE, lw=1)
    ax.set_axisbelow(True)
    for s in ("top", "right", "bottom", "left"):
        ax.spines[s].set_visible(False)
    ax.tick_params(length=0, colors=MUTED, labelsize=8.5)


def plot_loss_curves(hist, zoom_from=20, figsize=(11.6, 4.5)):
    """Two panels: the whole run, and a zoom on the converged region.

    The full curve is dominated by the first ~15 epochs, which flattens everything
    after into a single line — so the zoom is where the train/val gap, the best-val
    epoch and any overfitting actually become readable.
    """
    tr, va = np.array(hist["train"]), np.array(hist["val"])
    ep = np.arange(1, len(tr) + 1)
    best = int(np.argmin(va))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    # ---- left: the whole run
    ax1.plot(ep, tr, color=TRAIN, lw=2, label="training", solid_capstyle="round")
    ax1.plot(ep, va, color=VAL, lw=2, label="validation", solid_capstyle="round")
    ax1.axvspan(zoom_from, len(tr), color=INK, alpha=.05, lw=0)
    ax1.annotate("zoomed →", (zoom_from + 2, tr.max() * .82), fontsize=8.5,
                 color=FAINT, family="monospace")
    ax1.set_xlim(0, len(tr) + 1)
    ax1.set_title("the whole run", fontsize=11, fontweight="bold", color=INK, loc="left", pad=10)
    ax1.set_ylabel("class-weighted cross-entropy — lower is better",
                   fontsize=8.5, color=MUTED, family="monospace", labelpad=9)
    ax1.legend(frameon=False, fontsize=9.5, labelcolor=MUTED, handlelength=1.6)
    _style(ax1)

    # ---- right: converged region, where the gap is legible
    m = ep >= zoom_from
    ax2.fill_between(ep[m], tr[m], va[m], color=VAL, alpha=.13, lw=0)
    ax2.plot(ep[m], tr[m], color=TRAIN, lw=2, solid_capstyle="round")
    ax2.plot(ep[m], va[m], color=VAL, lw=2, solid_capstyle="round")
    ax2.scatter([ep[best]], [va[best]], s=54, color=VAL, zorder=5,
                edgecolor="white", linewidth=1.8)
    # both labels go inside the shaded gap, clear of either curve
    ax2.annotate(f"best val {va[best]:.4f}\nepoch {ep[best]}", (ep[best], va[best]),
                 textcoords="offset points", xytext=(-16, -34), ha="right", va="top",
                 fontsize=8.5, color=MUTED,
                 arrowprops=dict(arrowstyle="-", color=RULE, lw=1))
    mid = int(len(ep) * 0.45)                      # where the gap is widest
    gap = va[best] - tr[best]
    ax2.annotate(f"generalisation gap {gap:+.3f}", (ep[mid], (tr[mid] + va[mid]) / 2),
                 ha="center", va="center", fontsize=8.5, color=VAL, family="monospace",
                 bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=.75))
    ax2.set_xlim(zoom_from, len(tr) + 1)
    ax2.set_title(f"from epoch {zoom_from}", fontsize=11, fontweight="bold",
                  color=INK, loc="left", pad=10)
    _style(ax2)

    for ax in (ax1, ax2):
        ax.set_xlabel("epoch", fontsize=8.5, color=MUTED, family="monospace", labelpad=7)

    split = ("phrases held out — no transposed siblings shared"
             if hist.get("strategy") == "grouped"
             else f"random over the augmented corpus — "
                  f"{hist.get('shared_phrases','?')} of "
                  f"{hist.get('n_train_phrases','?')} phrases on both sides")
    fig.suptitle(f"{hist['role']}  ·  {hist['n_train']:,} train / {hist['n_val']:,} val rows"
                 f"  ·  split: {split}",
                 fontsize=9.5, color=MUTED, family="monospace", y=1.02, x=.02, ha="left")
    fig.subplots_adjust(wspace=.16)
    return fig


def gap_summary(hist):
    tr, va = np.array(hist["train"]), np.array(hist["val"])
    b = int(np.argmin(va))
    return {"best_val": round(float(va[b]), 4), "best_epoch": b + 1,
            "train_at_best": round(float(tr[b]), 4),
            "gap_at_best": round(float(va[b] - tr[b]), 4),
            "final_train": round(float(tr[-1]), 4), "final_val": round(float(va[-1]), 4),
            "val_rise_after_best": round(float(va[-1] - va[b]), 4)}

## 3. Render

In [ ]:
fig = plot_loss_curves(hist, zoom_from=20)
fig.savefig("figures/loss-curves-bass.png", dpi=170,
            bbox_inches="tight", facecolor="white")
print(json.dumps(gap_summary(hist), indent=1))
fig;

## 4. Reading it — and one caveat that matters

Loss falls fast: most of the drop happens in the first 15 epochs, and from about
epoch 40 the curve is grinding out small improvements. Validation bottoms at epoch
131 and drifts up slightly afterwards — mild overfitting, and the point
`train_notes.py` would have checkpointed.

**But this validation split does not measure generalisation.** `train_notes.py` calls
`random_split` on the corpus *after* 12-key augmentation, so a phrase's transposed
siblings land on both sides of the line. In this run **300 of 354 phrases appear in
both train and validation** — every validation phrase has siblings the model trained
on, in another key.

Since the augmentation exists precisely to make the model key-invariant, a transposed
sibling is very nearly the same example from the model's point of view. So the gap of
+0.022 is close to a best case, and the real gap on unseen phrases is likely wider.

To measure that instead, rerun with `strategy="grouped"`, which assigns whole phrases
to one side so all 12 keys follow together.

In [ ]:
# ~7.5 min. Uncomment to run the stricter split and compare.
# strict = train_with_history(role="bass", strategy="grouped", epochs=150)
# json.dump(strict, open("experiments/loss-curves/history_bass_grouped.json","w"), indent=1)
# fig2 = plot_loss_curves(strict, zoom_from=20)
# print('augmented split:', gap_summary(hist))
# print('grouped split:  ', gap_summary(strict))